# Problem Resolution Rate Analysis
# Analyse du Taux de Résolution des Problèmes

**🇬🇧** This notebook analyzes how users resolve problems across difficulty levels (A–F) and proficiency groups (G1–G6). The unit of analysis is the **(user, problem) pair**: for each pair, we track whether the user eventually reached AC, how many attempts it took, and whether they solved it on the first try.

**🇫🇷** Ce notebook analyse la manière dont les utilisateurs résolvent les problèmes en fonction du niveau de difficulté (A–F) et du groupe de proficience (G1–G6). L'unité d'analyse est la **paire (utilisateur, problème)** : pour chaque paire, on étudie si l'utilisateur a finalement obtenu AC, en combien de tentatives, et s'il a réussi du premier coup.

---

**CSVs utilisés :** `data/processed/atcoder_resolution_by_difficulty.csv`, `atcoder_resolution_by_group.csv`

**Structure :**
- **S1** : Taux de résolution, d'abandon et de réussite au premier essai par difficulté — vue d'ensemble / Resolution, abandon, and first-try rates by difficulty
- **S2** : Nombre moyen de tentatives avant AC par difficulté / Average attempts before AC by difficulty
- **S3** : Heatmap du taux de résolution par difficulté × groupe — montre l'effet croisé du niveau du problème et de la compétence / Resolution rate heatmap
- **S4** : Taux de réussite au premier essai par difficulté × groupe / First-try AC rate by difficulty × group
- **S5** : Nombre moyen de tentatives avant AC par difficulté × groupe / Average attempts before AC by difficulty × group


In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

DIFFICULTY_ORDER = ['A', 'B', 'C', 'D', 'E', 'F']
GROUP_ORDER      = ['G1', 'G2', 'G3', 'G4', 'G5', 'G6']

# Palette G1→G6 (clair → foncé)
PALETTE = ['#d4e6f1', '#85c1e9', '#3498db', '#1a5276', '#154360', '#0a1f33']

# Load pipeline outputs
df_diff  = pl.read_csv('../data/processed/atcoder_resolution_by_difficulty.csv')
df_group = pl.read_csv('../data/processed/atcoder_resolution_by_group.csv')

print('Resolution by difficulty:', df_diff.shape)
print('Resolution by group:     ', df_group.shape)
print()
print(df_diff.sort('difficulty'))

---
## Section 1 — Resolution, Abandon & First Try Rates by Difficulty
## Section 1 — Taux de Résolution, d'Abandon et de Réussite au Premier Essai par Difficulté

**Why this figure:** Three rates tell three different stories about how users engage with problems. The resolution rate shows persistence, the abandon rate shows discouragement, and the first try rate measures mastery. Plotting them together across A–F reveals how difficulty progressively challenges users along all three dimensions.

**Pourquoi cette figure :** Trois taux racontent trois histoires différentes sur la manière dont les utilisateurs s'engagent avec les problèmes. Le taux de résolution mesure la persistance, le taux d'abandon mesure le découragement, et le taux de réussite au premier essai mesure la maîtrise. Les afficher ensemble sur A–F montre comment la difficulté défie progressivement les utilisateurs sur ces trois dimensions.

---

In [ ]:
df_d = df_diff.sort('difficulty').to_pandas()

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(DIFFICULTY_ORDER))
w = 0.26

b1 = ax.bar(x - w,   df_d['resolution_rate'], w, label='Resolved / Résolu',         color='#27ae60', edgecolor='white')
b2 = ax.bar(x,       df_d['first_try_rate'],   w, label='First Try AC / 1er essai',  color='#3498db', edgecolor='white')
b3 = ax.bar(x + w,   df_d['abandon_rate'],     w, label='Abandoned / Abandonné',     color='#e74c3c', edgecolor='white')

for bar, v in [(b, val) for bars, vals in [(b1, df_d['resolution_rate']), (b2, df_d['first_try_rate']), (b3, df_d['abandon_rate'])] for b, val in zip(bars, vals)]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(DIFFICULTY_ORDER)
ax.set_xlabel('Difficulty Level / Niveau de difficulté', fontsize=12)
ax.set_ylabel('Rate (%) / Taux (%)', fontsize=12)
ax.set_title(
    'Resolution, First Try & Abandon Rates by Difficulty Level\n'
    'Taux de résolution, premier essai et abandon par niveau de difficulté',
    fontsize=13, pad=12
)
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Raw data / Données brutes:')
print(df_d[['difficulty', 'n_pairs', 'resolution_rate', 'abandon_rate', 'first_try_rate']].to_string(index=False))

### Analysis / Analyse

**🇫🇷** Le taux de résolution finale reste haut sur toute la plage A–F, ne descendant qu'à 80.5% sur le niveau F. Ce plancher s'explique en partie par un biais de sélection : les utilisateurs qui tentent des problèmes F sont déjà un sous-ensemble très filtré.

Les deux taux racontent des histoires légèrement différentes. Pour la *résolution finale*, le plus grand saut est en C→D (-6 pp), nettement au-dessus des autres transitions (~3 pp sur D–F, ~1 pp sur A→B). Pour le *premier essai*, l'inflexion la plus forte est en B→C (-13.8 pp), suivie de C→D (-10 pp), puis une baisse régulière d'environ 6 pp par niveau sur D–F. Les deux courbes, ainsi que celle de l'abandon (rouge), convergent vers le même diagnostic : la zone B–D constitue le seuil où la difficulté conceptuelle prend le dessus.

---

**🇬🇧** The overall resolution rate stays high across A–F, only dropping to 80.5% at level F. This floor is partly explained by selection bias: users who attempt F problems are already a highly filtered subset.

The two rates tell slightly different stories. For the *final resolution rate*, the largest drop is at C→D (-6 pp), well above the other transitions (~3 pp across D–F, ~1 pp at A→B). For the *first try rate*, the sharpest inflection is at B→C (-13.8 pp), followed by C→D (-10 pp), then a steady decline of about 6 pp per level from D to F. Both curves, along with the abandon rate (red), converge on the same conclusion: the B–D zone is where conceptual difficulty takes over.

---
## Section 2 — Average Attempts to AC by Difficulty
## Section 2 — Nombre Moyen de Tentatives avant AC par Difficulté

**Why this figure:** The average number of submissions before the first AC (for users who eventually succeeded) captures how much iteration is required to solve a problem. This is independent of the error type — it reflects the overall trial-and-error cost per difficulty level.

**Pourquoi cette figure :** Le nombre moyen de soumissions avant le premier AC (uniquement pour les utilisateurs qui ont réussi) capture l'effort d'itération nécessaire pour résoudre un problème. C'est indépendant du type d'erreur — cela reflète le coût global d'essai-erreur par niveau de difficulté.

---

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#85c1e9', '#5dade2', '#3498db', '#2980b9', '#1a5276', '#154360']
bars = ax.bar(DIFFICULTY_ORDER, df_d['avg_attempts_to_ac'], color=colors, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, df_d['avg_attempts_to_ac']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{v:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Difficulty Level / Niveau de difficulté', fontsize=12)
ax.set_ylabel('Avg submissions to first AC / Soumissions moyennes avant AC', fontsize=12)
ax.set_title(
    'Average Number of Submissions to First AC by Difficulty Level (resolved pairs only)\n'
    'Nombre moyen de soumissions avant le premier AC par niveau (paires résolues uniquement)',
    fontsize=13, pad=12
)
ax.set_ylim(0, df_d['avg_attempts_to_ac'].max() + 0.4)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** Le saut B→C (+0.38) est le plus grand de la séquence et confirme l'inflexion observée en figure 1. Les incréments suivants sont plus faibles (D→E : +0.26, E→F : +0.19) sans qu'on puisse en tirer une conclusion forte.

Les valeurs pour A (1.30) et B (1.48) semblent particulièrement basses : elles sont tirées vers le bas par les groupes avancés (G4–G6) qui tentent aussi ces niveaux faciles et les résolvent presque toujours du premier coup. La valeur sur F (2.59) est la plus haute de la séquence mais reste modeste, car les utilisateurs qui tentent des problèmes F sont déjà forts en algorithmique et itèrent peu. La figure 5 donnera une vision par groupe qui permettra de lire ces chiffres sans ce biais.

---

**🇬🇧** The B→C jump (+0.38) is the largest in the sequence and confirms the inflection observed in figure 1. Subsequent increments are smaller (D→E: +0.26, E→F: +0.19), without a strong conclusion to draw from them.

The values for A (1.30) and B (1.48) appear particularly low: they are pulled down by advanced groups (G4–G6) who also attempt these easy levels and almost always solve them on the first try. The value at F (2.59) is the highest in the sequence but remains modest, as users who attempt F problems are already strong algorithmically and iterate little. Figure 5 will provide a per-group breakdown that allows reading these numbers without this bias.

---
## Section 3 — Resolution Rate Heatmap by Difficulty × Group
## Section 3 — Heatmap du Taux de Résolution par Difficulté × Groupe

**Why this figure:** This heatmap directly shows the combined effect of problem difficulty and user proficiency on resolution success. It mirrors the AC rate heatmap from notebook 03, but here resolution means *eventually reaching AC across all attempts*, not just on a single submission. Comparing both heatmaps reveals whether expert users compensate for harder problems through persistence.

**Pourquoi cette figure :** Cette heatmap montre l'effet combiné de la difficulté du problème et de la proficience de l'utilisateur sur le succès de résolution. Elle fait écho à la heatmap de taux AC du notebook 03, mais ici la résolution signifie *atteindre AC sur l'ensemble des tentatives*, pas seulement sur une seule soumission. Comparer les deux heatmaps révèle si les experts compensent les problèmes difficiles par la persistance.

---

In [ ]:
def build_matrix(metric):
    mat = pd.DataFrame(index=DIFFICULTY_ORDER, columns=GROUP_ORDER, dtype=float)
    for diff in DIFFICULTY_ORDER:
        for g in GROUP_ORDER:
            subset = df_group.filter(
                (pl.col('difficulty') == diff) & (pl.col('proficiency_group') == g)
            )
            mat.loc[diff, g] = subset[metric].item() if not subset.is_empty() else np.nan
    return mat.astype(float)

resolution_matrix = build_matrix('resolution_rate')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    resolution_matrix,
    annot=True, fmt='.1f', cmap='Greens',
    vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    mask=resolution_matrix.isna(),
    cbar_kws={'label': 'Resolution rate (%)'},
    ax=ax
)
ax.set_title(
    'Resolution Rate (%) by Difficulty Level × Proficiency Group\n'
    'Taux de résolution (%) par niveau de difficulté × groupe de proficience',
    fontsize=13, pad=12
)
ax.set_xlabel('Proficiency Group / Groupe de proficience', fontsize=12)
ax.set_ylabel('Difficulty Level / Niveau de difficulté', fontsize=12)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** Chaque groupe atteint son taux de résolution le plus bas sur son niveau max : G4/D (72.0%), G5/E (72.5%), G6/F (86.6%). Cette tendance présente toutefois une exception notable : G6 affiche un taux bien plus élevé que G4 et G5 sur leurs niveaux respectifs, révélant une maîtrise plus robuste. Sur le niveau A, le gradient horizontal est déjà visible : G1 résout 91.6% des problèmes A contre 99.5% pour G6, soit 8 pp d'écart sur les problèmes les plus simples.

---

**🇬🇧** Each group reaches its lowest resolution rate at its maximum difficulty level: G4/D (72.0%), G5/E (72.5%), G6/F (86.6%). This trend has a notable exception, however: G6 achieves a much higher rate than G4 and G5 at their respective ceiling levels, reflecting more robust mastery. Even on level A, the horizontal gradient is visible: G1 resolves 91.6% of A problems versus 99.5% for G6, an 8 pp gap on the easiest problems.

---
## Section 4 — First Try AC Rate by Difficulty × Group
## Section 4 — Taux de Réussite au Premier Essai par Difficulté × Groupe

**Why this figure:** First Try AC is the strongest indicator of mastery — the user submitted correct code immediately, with no prior failed attempt on that problem. Unlike the resolution rate, it is not inflated by persistence and reflects genuine preparation and skill. The contrast between G1 and G6 on this metric should be dramatic.

**Pourquoi cette figure :** La réussite au premier essai est l'indicateur de maîtrise le plus fort — l'utilisateur a soumis un code correct immédiatement, sans tentative échouée préalable. Contrairement au taux de résolution, il n'est pas gonflé par la persistance et reflète une vraie préparation et compétence. Le contraste entre G1 et G6 sur cette métrique devrait être marqué.

---

In [ ]:
first_try_matrix = build_matrix('first_try_rate')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    first_try_matrix,
    annot=True, fmt='.1f', cmap='Blues',
    vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    mask=first_try_matrix.isna(),
    cbar_kws={'label': 'First Try AC rate (%)'},
    ax=ax
)
ax.set_title(
    'First Try AC Rate (%) by Difficulty Level × Proficiency Group\n'
    'Taux de réussite au premier essai (%) par niveau de difficulté × groupe de proficience',
    fontsize=13, pad=12
)
ax.set_xlabel('Proficiency Group / Groupe de proficience', fontsize=12)
ax.set_ylabel('Difficulty Level / Niveau de difficulté', fontsize=12)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** Le taux de premier essai décroît plus vite que le taux de résolution lorsque la difficulté augmente : l'écart entre "finir par réussir" et "réussir du premier coup" se creuse. Sur le niveau D, G4 résout 72% des problèmes mais seulement ~35% du premier coup.

À l'autre extrême, G6 résout 42% des problèmes F dès la première soumission : près d'un problème F sur deux sans essai raté, ce qui traduit une vraie maîtrise algorithmique. Il est aussi notable que G1 réussit 57.1% des problèmes A du premier coup, soit plus d'un sur deux, même pour les utilisateurs les moins avancés. Même logique pour G2 sur les problèmes B. Cela montre que les niveaux A et B sont conçus pour être accessibles. Sur le niveau A, l'écart G1 (57.1%) vs G6 (87.4%) de 30 pp montre néanmoins que l'expertise influence profondément la qualité de la première soumission.

---

**🇬🇧** The first try rate decays faster than the resolution rate as difficulty increases: the gap between "eventually resolving" and "resolving on the first attempt" widens. At level D, G4 resolves 72% of problems but only ~35% on the first try.

At the other extreme, G6 resolves 42% of F problems on the first submission — nearly one in two F problems without a failed attempt, reflecting genuine algorithmic mastery. It is also worth noting that G1 succeeds on 57.1% of A problems on the first try, more than one in two, even for the least advanced users. The same pattern holds for G2 on B problems. This suggests levels A and B are designed to be accessible. On level A, however, the G1 (57.1%) vs G6 (87.4%) gap of 30 pp shows that expertise still deeply influences first-submission quality.

---
## Section 5 — Average Attempts to AC by Difficulty × Group
## Section 5 — Nombre Moyen de Tentatives avant AC par Difficulté × Groupe

**Why this figure:** For users who eventually resolve a problem, how many attempts does it take? This heatmap shows whether experts need fewer iterations than beginners at the same difficulty, and how that gap evolves from A to F.

**Pourquoi cette figure :** Pour les utilisateurs qui finissent par résoudre un problème, combien de tentatives leur faut-il ? Cette heatmap montre si les experts ont besoin de moins d'itérations que les débutants à difficulté égale, et comment cet écart évolue de A à F.

---

In [ ]:
attempts_matrix = build_matrix('avg_attempts_to_ac')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    attempts_matrix,
    annot=True, fmt='.2f', cmap='Oranges',
    vmin=1, vmax=attempts_matrix.max().max(),
    linewidths=0.5, linecolor='white',
    mask=attempts_matrix.isna(),
    cbar_kws={'label': 'Avg attempts to first AC'},
    ax=ax
)
ax.set_title(
    'Average Attempts to First AC by Difficulty Level × Proficiency Group (resolved pairs only)\n'
    'Tentatives moyennes avant le premier AC par niveau × groupe (paires résolues uniquement)',
    fontsize=13, pad=12
)
ax.set_xlabel('Proficiency Group / Groupe de proficience', fontsize=12)
ax.set_ylabel('Difficulty Level / Niveau de difficulté', fontsize=12)
plt.tight_layout()
plt.show()

### Analysis / Analyse

**🇫🇷** En cohérence avec les figures précédentes, chaque groupe peine le plus sur son niveau max (G4/D : 2.53, G5/E : 2.65). Le gradient horizontal à difficulté constante confirme l'avantage expert : sur le niveau D, G4 a besoin de 2.53 tentatives contre 1.98 pour G6, soit 22% de tentatives en moins. L'avantage expert ne se limite donc pas à un taux de résolution plus élevé (figure 3), il se traduit aussi par une résolution plus efficace.

Cette figure éclaire aussi la figure 2 : la moyenne globale sur le niveau A (1.30) est bien inférieure à celle de G1 sur A (1.94), car les groupes forts tirent la moyenne vers le bas. Les valeurs de la figure 2 doivent donc être lues comme des moyennes pondérées par la composition du pool, et non comme l'expérience d'un utilisateur typique.

En combinant figures 3, 4 et 5 : G6 résout 86.6% des problèmes F, dont 42% du premier coup, avec 2.59 tentatives en moyenne sur l'ensemble des paires résolues. Ce portrait confirme que la classification G6 capture une expertise réelle : ces utilisateurs connaissent soit la solution d'emblée, soit convergent très rapidement vers elle.

---

**🇬🇧** Consistent with previous figures, each group struggles most at its maximum difficulty level (G4/D: 2.53, G5/E: 2.65). The horizontal gradient at constant difficulty confirms the expert advantage: at level D, G4 needs 2.53 attempts versus 1.98 for G6, 22% fewer attempts. The expert advantage is therefore not limited to a higher resolution rate (figure 3), it also translates into more efficient resolution.

This figure also sheds light on figure 2: the global average for level A (1.30) is well below G1's average on A (1.94), because stronger groups pull the mean down. The values in figure 2 should therefore be read as pool-weighted averages, not as the experience of a typical user.

Combining figures 3, 4 and 5: G6 resolves 86.6% of F problems, 42% on the first try, with an average of 2.59 attempts across all resolved pairs. This portrait confirms that the G6 classification captures genuine expertise: these users either know the solution immediately or converge to it very quickly.